# Intro to pandas

This notebook has been set up to cover some introductory concepts in pandas for CCC colleagues.

### Packages

A reasonable amount of functionality comes built in to Python, covering everything we did in the 'intro to python' session and more. However, one of the great things about Python (and other languages, like R (and KNIME)) is that they have an active user base who contribute packages. Packages are add-on pieces of functionality which have been designed to help with particular (or quite general) problems. 

### Pandas

One of the most popular packages that we will certainly be using in our work is pandas, which provides a range of functionality for manipulating tabular data. Let's take a look.

In [1]:
# installing the pandas package

!pip install pandas

# installing another package which we will need later on

!pip install openpyxl

In [2]:
# now importing it into this notebook so we can use it

import pandas as pd

The central concept in pandas is something called a "dataframe" (often written as DataFrame per pandas syntax), which is basically a data table with a set of rows and columns, and a lot of integrated functionality. Dataframes can be created by reading in files (e.g. from Excel), which is what we're going to do to explore some basic functionality here.

We've got the cb7 full dataset saved in a data subfolder within this repository. Let's take a look at that using pandas.

In [ ]:
# we know our file has multiple tabs
# note use of pd. syntax, which indicates that we are using some pandas functionality

file = pd.ExcelFile("../data/cb7_full_dataset.xlsx")
file.sheet_names

['lists',
 'Contents',
 'Meta data >>',
 'Variable definitions',
 'Measure definitions',
 'Sector classification',
 'Data explorer >>',
 'Summary charts',
 'Summary data',
 'Raw data >>',
 'Economy-wide data',
 'Sector-level data',
 'Subsector-level data',
 'Measure-level data',
 'Alt. sector classifications']

In [4]:
# we don't want to look at all of these tabs now - let's just look at the sector-level data
# note that the term df is commonly used as a shorthand for DataFrame

df = pd.read_excel(file, sheet_name="Sector-level data")

### Understanding dataframes

Before we do anything at all, we need to understand some basics about the DataFrame structure. A DataFrame has the following key components:

1) Rows
2) Columns
3) An index

The first and second should be intuitive to anyone who has used Excel. The third might not be, but is important to understand for many operations in pandas.

In [5]:
# we've now got a dataframe
# Let's look at the first few rows to confirm it is what we want

df.head(5)

,scenario,country,sector,variable,variable_unit,year,value
0,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2025,47.044319
1,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2026,46.559718
2,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2027,44.919614
3,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2028,43.081191
4,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2029,41.070621


In [6]:
# looking at the column names

df.columns

Index(['scenario', 'country', 'sector', 'variable', 'variable_unit', 'year',
       'value'],
      dtype='object')

In [7]:
# looking at an individual column
# this is actually a pandas Series
# note use of square brackets to access individual columns

df["scenario"]

0        Balanced Pathway
1        Balanced Pathway
2        Balanced Pathway
3        Balanced Pathway
4        Balanced Pathway
               ...       
20379            Baseline
20380            Baseline
20381            Baseline
20382            Baseline
20383            Baseline
Name: scenario, Length: 20384, dtype: object

In [8]:
# looking at the index

df.index

RangeIndex(start=0, stop=20384, step=1)

Having a range index like this is probably not preferable for timeseries data, where we would generally want our time/date column to be the index for simplicity. We will address this later.

In [9]:
# we can also use some very general pandas methods to describe our dataset
# first the .info method, which gives us an overview

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20384 entries, 0 to 20383
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   scenario       20384 non-null  object 
 1   country        20384 non-null  object 
 2   sector         20384 non-null  object 
 3   variable       20384 non-null  object 
 4   variable_unit  20384 non-null  object 
 5   year           20384 non-null  int64  
 6   value          20384 non-null  float64
dtypes: float64(1), int64(1), object(5)
memory usage: 1.1+ MB


In [10]:
# we can use .describe to give us basic stats about the numeric columns

df.describe()

,year,value
count,20384.000000,20384.000000
mean,2037.500000,83.628217
std,7.500184,1516.868521
min,2025.000000,-23818.302471
25%,2031.000000,0.000000
50%,2037.500000,0.000000
75%,2044.000000,10.956378
max,2050.000000,28081.744094


In [11]:
# that looks about right
# we can use some other basic functionality to inspect individual columns

print(f"The data covers the following years: {df["year"].unique()}") # looking at the year-range

The data covers the following years: [2025 2026 2027 2028 2029 2030 2031 2032 2033 2034 2035 2036 2037 2038
 2039 2040 2041 2042 2043 2044 2045 2046 2047 2048 2049 2050]


In [12]:
# we can also use a loop to look at main entries in each columns:

for column in df.columns:
    print(column)
    print(df[column].unique())

scenario
['Balanced Pathway' 'Baseline']
country
['United Kingdom']
sector
['Agriculture' 'Aviation' 'Electricity supply' 'Engineered removals'
 'F-gases' 'Fuel supply' 'Industry' 'Land use' 'Non-residential buildings'
 'Residential buildings' 'Shipping' 'Surface transport' 'Waste']
variable
['Emissions: direct emissions total' 'Emissions: direct emissions CO2'
 'Emissions: direct emissions CH4' 'Emissions: direct emissions N2O'
 'Emissions: direct emissions F-gases' 'Emissions: direct abatement total'
 'Emissions: direct abatement CO2' 'Emissions: direct abatement CH4'
 'Emissions: direct abatement N2O' 'Emissions: direct abatement F-gases'
 'Emissions: CCS' 'Energy: final demand electricity'
 'Energy: final demand gas abated' 'Energy: final demand gas unabated'
 'Energy: final demand oil' 'Energy: final demand solid fuel'
 'Energy: final demand bioenergy' 'Energy: final demand non-bio waste'
 'Energy: final demand hydrogen' 'Energy: gross demand total'
 'Energy: gross demand electric

### Simple operations on dataframes

The above operations give us a pretty good idea of what is in our data. We can now look at doing some basic operations to transform it.

Let's start with filtering data, in the same way that you might in Excel.

In [13]:
# let's also save the results to a new dataframe to carry forward

cost_df = df.loc[df["variable"].str.contains("Cost")] # using .loc functionality to filter to only cost data

In [14]:
# let's inspect this new dataframe

cost_df

,scenario,country,sector,variable,variable_unit,year,value
884,Balanced Pathway,United Kingdom,Agriculture,Cost: additional capital expenditure (adjusted),£m,2025,339.679500
885,Balanced Pathway,United Kingdom,Agriculture,Cost: additional capital expenditure (adjusted),£m,2026,348.453716
886,Balanced Pathway,United Kingdom,Agriculture,Cost: additional capital expenditure (adjusted),£m,2027,267.850613
887,Balanced Pathway,United Kingdom,Agriculture,Cost: additional capital expenditure (adjusted),£m,2028,237.337657
888,Balanced Pathway,United Kingdom,Agriculture,Cost: additional capital expenditure (adjusted),£m,2029,227.156943
...,...,...,...,...,...,...,...
13281,Balanced Pathway,United Kingdom,Waste,Cost: additional capital expenditure annualise...,£m,2046,1011.313962
13282,Balanced Pathway,United Kingdom,Waste,Cost: additional capital expenditure annualise...,£m,2047,1004.938967
13283,Balanced Pathway,United Kingdom,Waste,Cost: additional capital expenditure annualise...,£m,2048,999.000806
13284,Balanced Pathway,United Kingdom,Waste,Cost: additional capital expenditure annualise...,£m,2049,992.903041


In [15]:
# let's look at what the values in this column are
# we might want to narrow this down more

cost_df["variable"].unique()

array(['Cost: additional capital expenditure (adjusted)',
       'Cost: additional operating expenditure (adjusted)',
       'Cost: additional capital expenditure (unadjusted)',
       'Cost: additional operating expenditure (unadjusted)',
       'Cost: additional capital expenditure annualised (adjusted)'],
      dtype=object)

We probably don't want both adjusted and unadjusted figures if we're going to do any analysis in a grouped way. Let's remove the unadjusted figures.


In [16]:
# using .loc again - note use of the ~ to indicate the inverse of our condition

cost_df = cost_df.loc[~cost_df["variable"].str.contains("unadjusted")]

cost_df["variable"].unique()

array(['Cost: additional capital expenditure (adjusted)',
       'Cost: additional operating expenditure (adjusted)',
       'Cost: additional capital expenditure annualised (adjusted)'],
      dtype=object)

We might want to change some of the data e.g. simplifying variable names.

In [17]:
# defining a renaming dictionary to replace the current entries with

renamer = {'Cost: additional capital expenditure (adjusted)' : 'capex',
           'Cost: additional operating expenditure (adjusted)' : 'opex',
           'Cost: additional capital expenditure annualised (adjusted)' : 'annualised_capex'}

cost_df["variable"] = cost_df["variable"].replace(renamer)

cost_df

C:\Users\fw000011\AppData\Local\Temp\ipykernel_22976\2123288654.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cost_df["variable"] = cost_df["variable"].replace(renamer)


,scenario,country,sector,variable,variable_unit,year,value
884,Balanced Pathway,United Kingdom,Agriculture,capex,£m,2025,339.679500
885,Balanced Pathway,United Kingdom,Agriculture,capex,£m,2026,348.453716
886,Balanced Pathway,United Kingdom,Agriculture,capex,£m,2027,267.850613
887,Balanced Pathway,United Kingdom,Agriculture,capex,£m,2028,237.337657
888,Balanced Pathway,United Kingdom,Agriculture,capex,£m,2029,227.156943
...,...,...,...,...,...,...,...
13281,Balanced Pathway,United Kingdom,Waste,annualised_capex,£m,2046,1011.313962
13282,Balanced Pathway,United Kingdom,Waste,annualised_capex,£m,2047,1004.938967
13283,Balanced Pathway,United Kingdom,Waste,annualised_capex,£m,2048,999.000806
13284,Balanced Pathway,United Kingdom,Waste,annualised_capex,£m,2049,992.903041


### More advanced operations

One thing we will probably often want to do to data is group it in some way (analogous to SUMIF, AVERAGEIF etc in Excel). Let's create a new dataframe which stores the results of a grouping operation.

In [18]:
# Note the specific syntax here: selecting columns, .groupby(), .sum()

grouped_costs_df = cost_df[["year", "variable", "value"]].groupby(by=["year", "variable"]).sum()

grouped_costs_df

value
year variable                      
2025 annualised_capex   2228.472513
     capex             16929.750728
     opex              -2309.870536
2026 annualised_capex   3855.956246
     capex             27725.348958
...                             ...
2049 capex              1671.202899
     opex             -32351.536097
2050 annualised_capex  29935.416792
     capex             -2277.435499
     opex             -32996.966445

[78 rows x 1 columns]

Another key operation is reshaping our data. If we have long data (as we do here), we might want to pivot it to get the entries of one column as new columns. The inverse operation also exists, using the .melt() method in pandas.

In [19]:
# we can also do pivot operations to change the shape of the data
# NOTE: WE'RE SETTING AN INDEX IN THE BELOW OPERATION. THIS WILL BE RELEVANT LATER ON.

pivot_df = grouped_costs_df.reset_index().pivot(columns="variable", values="value", index="year")

pivot_df

variable,annualised_capex,capex,opex
year,,,
2025,2228.472513,16929.750728,-2309.870536
2026,3855.956246,27725.348958,-5835.026909
2027,5816.394854,32837.756634,-8746.552268
2028,8464.145284,43400.458669,-11079.493624
2029,11302.700772,45995.451856,-13044.872392
2030,13919.340011,44807.386739,-14856.218025
2031,16555.227324,42139.286245,-15808.339981
2032,19036.003182,39113.227326,-17299.408456
2033,21473.747254,39089.742118,-18615.194672


In [20]:
# we can sort the data like you would in excel if that's of interest

pivot_df.sort_values(by="capex", ascending=False)

variable,annualised_capex,capex,opex
year,,,
2029,11302.700772,45995.451856,-13044.872392
2030,13919.340011,44807.386739,-14856.218025
2028,8464.145284,43400.458669,-11079.493624
2035,26341.487936,42167.098617,-20577.571262
2031,16555.227324,42139.286245,-15808.339981
2036,28510.896060,39961.645181,-20742.647142
2032,19036.003182,39113.227326,-17299.408456
2033,21473.747254,39089.742118,-18615.194672
2034,23870.128910,39017.298348,-19708.920693


In [21]:
# note that sorting operation has not been saved!

pivot_df

variable,annualised_capex,capex,opex
year,,,
2025,2228.472513,16929.750728,-2309.870536
2026,3855.956246,27725.348958,-5835.026909
2027,5816.394854,32837.756634,-8746.552268
2028,8464.145284,43400.458669,-11079.493624
2029,11302.700772,45995.451856,-13044.872392
2030,13919.340011,44807.386739,-14856.218025
2031,16555.227324,42139.286245,-15808.339981
2032,19036.003182,39113.227326,-17299.408456
2033,21473.747254,39089.742118,-18615.194672


A fundamental operation we haven't explored yet - creating new columns!

In [22]:
# note syntax - we create a new column simply by writing its name and assigning results

pivot_df["net_annual_cost"] = pivot_df["annualised_capex"] + pivot_df["opex"] 

In [23]:
# looking at that again

pivot_df

variable,annualised_capex,capex,opex,net_annual_cost
year,,,,
2025,2228.472513,16929.750728,-2309.870536,-81.398023
2026,3855.956246,27725.348958,-5835.026909,-1979.070662
2027,5816.394854,32837.756634,-8746.552268,-2930.157414
2028,8464.145284,43400.458669,-11079.493624,-2615.348340
2029,11302.700772,45995.451856,-13044.872392,-1742.171620
2030,13919.340011,44807.386739,-14856.218025,-936.878014
2031,16555.227324,42139.286245,-15808.339981,746.887343
2032,19036.003182,39113.227326,-17299.408456,1736.594726
2033,21473.747254,39089.742118,-18615.194672,2858.552582


In [24]:
# we might want to categorise a year as saving/non-saving

pivot_df["saving"] = pivot_df["net_annual_cost"].apply(lambda x: True if x < 0 else False)
pivot_df

variable,annualised_capex,capex,opex,net_annual_cost,saving
year,,,,,
2025,2228.472513,16929.750728,-2309.870536,-81.398023,True
2026,3855.956246,27725.348958,-5835.026909,-1979.070662,True
2027,5816.394854,32837.756634,-8746.552268,-2930.157414,True
2028,8464.145284,43400.458669,-11079.493624,-2615.348340,True
2029,11302.700772,45995.451856,-13044.872392,-1742.171620,True
2030,13919.340011,44807.386739,-14856.218025,-936.878014,True
2031,16555.227324,42139.286245,-15808.339981,746.887343,False
2032,19036.003182,39113.227326,-17299.408456,1736.594726,False
2033,21473.747254,39089.742118,-18615.194672,2858.552582,False


We don't have to do everything in columns within our dataframe. Let's save some variables for key statistics.

In [25]:
# we can pick out the years with the highest and lowest net cost

max_spend = pivot_df["net_annual_cost"].max()
max_spend_year = pivot_df[pivot_df["net_annual_cost"] == max_spend].index.values[0]

# and total cost across entire period

total_cost = pivot_df["net_annual_cost"].sum()

print(f"""The year with highest net cost is {max_spend_year}, with costs of £m{max_spend:,.0f}.
The net total cost is £m {total_cost:,.0f}""")

The year with highest net cost is 2040, with costs of £m10,049.
The net total cost is £m 88,773


### Joining and merging

This is all great, but what happens when we have another dataset that we want to bring in to our analysis. Joining and merging datasets is a key operation that we will encounter a lot when working with data.

Let's take a look at the GDP forecast dataset for our example.

In [ ]:
# we have another dataset, which is the gdp forecast to 2050 used in CB7
# let's read this in and then do some more operations

gdp_df = pd.read_csv("../data/uk_gdp_forecast.csv")

gdp_df

,year,uk_gdp
0,2021,2142738.000
1,2022,2230625.000
2,2023,2226067.103
3,2024,2265898.161
4,2025,2321564.431
5,2026,2370946.703
6,2027,2415517.257
7,2028,2458985.548
8,2029,2503939.628
9,2030,2550941.076


We can see that this looks about right for the dataframe that we have been working with. The GDP dataframe has a column with values, and another with years. We should be able to join these together fairly easily right?

In [27]:
# let's do a join operation so that we can get this in a dataframe with our other data

new_df = pivot_df.join(gdp_df)

In [28]:
# looking at this new dataframe

new_df

,annualised_capex,capex,opex,net_annual_cost,saving,year,uk_gdp
year,,,,,,,
2025,2228.472513,16929.750728,-2309.870536,-81.398023,True,NaN,NaN
2026,3855.956246,27725.348958,-5835.026909,-1979.070662,True,NaN,NaN
2027,5816.394854,32837.756634,-8746.552268,-2930.157414,True,NaN,NaN
2028,8464.145284,43400.458669,-11079.493624,-2615.348340,True,NaN,NaN
2029,11302.700772,45995.451856,-13044.872392,-1742.171620,True,NaN,NaN
2030,13919.340011,44807.386739,-14856.218025,-936.878014,True,NaN,NaN
2031,16555.227324,42139.286245,-15808.339981,746.887343,False,NaN,NaN
2032,19036.003182,39113.227326,-17299.408456,1736.594726,False,NaN,NaN
2033,21473.747254,39089.742118,-18615.194672,2858.552582,False,NaN,NaN


This doesn't look like it's returned the right thing. Why not?

Because our two dataframes have different indexes, so they won't join automatically.

In [29]:
# that hasn't worked! Why not?
# the issue is to do with indexes

print(pivot_df.index)
print(gdp_df.index)

Index([2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036,
       2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2048,
       2049, 2050],
      dtype='int64', name='year')
RangeIndex(start=0, stop=30, step=1)


In [30]:
# let's set the gdp_df index to year and try again

gdp_df = gdp_df.set_index("year")

new_df = pivot_df.join(gdp_df)

In [31]:
new_df

,annualised_capex,capex,opex,net_annual_cost,saving,uk_gdp
year,,,,,,
2025,2228.472513,16929.750728,-2309.870536,-81.398023,True,2321564.431
2026,3855.956246,27725.348958,-5835.026909,-1979.070662,True,2370946.703
2027,5816.394854,32837.756634,-8746.552268,-2930.157414,True,2415517.257
2028,8464.145284,43400.458669,-11079.493624,-2615.348340,True,2458985.548
2029,11302.700772,45995.451856,-13044.872392,-1742.171620,True,2503939.628
2030,13919.340011,44807.386739,-14856.218025,-936.878014,True,2550941.076
2031,16555.227324,42139.286245,-15808.339981,746.887343,False,2598061.108
2032,19036.003182,39113.227326,-17299.408456,1736.594726,False,2645094.843
2033,21473.747254,39089.742118,-18615.194672,2858.552582,False,2692348.901


You might notice another particularity in our joined dataframe. Some of the years from the GDP dataset are missing. This is because of the type of join we're using, which is a 'left' join by default. Let's try another type and see what we get.

In [32]:
# looking at the below, we see that the data is only shown from 2025 onwards, even thought the uk_gdp series starts in 2021
# the type of join we do here is key!

new_df = pivot_df.join(gdp_df, how="outer")

new_df

,annualised_capex,capex,opex,net_annual_cost,saving,uk_gdp
year,,,,,,
2021,NaN,NaN,NaN,NaN,NaN,2142738.000
2022,NaN,NaN,NaN,NaN,NaN,2230625.000
2023,NaN,NaN,NaN,NaN,NaN,2226067.103
2024,NaN,NaN,NaN,NaN,NaN,2265898.161
2025,2228.472513,16929.750728,-2309.870536,-81.398023,True,2321564.431
2026,3855.956246,27725.348958,-5835.026909,-1979.070662,True,2370946.703
2027,5816.394854,32837.756634,-8746.552268,-2930.157414,True,2415517.257
2028,8464.145284,43400.458669,-11079.493624,-2615.348340,True,2458985.548
2029,11302.700772,45995.451856,-13044.872392,-1742.171620,True,2503939.628


And then inversing the order of the join, but with yet another type.

In [33]:
new_df = gdp_df.join(pivot_df, how="right")

In [34]:
new_df

,uk_gdp,annualised_capex,capex,opex,net_annual_cost,saving
year,,,,,,
2025,2321564.431,2228.472513,16929.750728,-2309.870536,-81.398023,True
2026,2370946.703,3855.956246,27725.348958,-5835.026909,-1979.070662,True
2027,2415517.257,5816.394854,32837.756634,-8746.552268,-2930.157414,True
2028,2458985.548,8464.145284,43400.458669,-11079.493624,-2615.348340,True
2029,2503939.628,11302.700772,45995.451856,-13044.872392,-1742.171620,True
2030,2550941.076,13919.340011,44807.386739,-14856.218025,-936.878014,True
2031,2598061.108,16555.227324,42139.286245,-15808.339981,746.887343,False
2032,2645094.843,19036.003182,39113.227326,-17299.408456,1736.594726,False
2033,2692348.901,21473.747254,39089.742118,-18615.194672,2858.552582,False


In the end, it's probably best to just have the data for 2025 onwards, so let's keep that.

### Bonus: graphs

There are lots of libraries in python to visualise data very nicely. There is a slight learning curve to making plots look nice in many libraries though, which we won't go into right now. However, using basic plotting can be very helpful for sense checking our data. Let's do that now.

In [35]:
!pip install matplotlib

In [37]:
import matplotlib.pyplot as plt
from utils import theme

fig, ax = plt.subplots(figsize=(9,6))

ax.plot(new_df["opex"], label="Operating expenditure")
ax.plot(new_df["annualised_capex"], label="Annualised capital expenditure")
ax.plot(new_df["net_annual_cost"], label="Net annual cost")
ax.set_ylabel("Cost (£m)")
ax.axhline(y=0, color="black")
ax.legend()

plt.show()

ModuleNotFoundError: No module named 'utils'